In [4]:
!pip install -q sentence-transformers
!pip install -q wikipedia-api
!pip install -q numpy
!pip install -q scipy

In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("Alibaba-NLP/gte-base-en-v1.5", trust_remote_code=True)

/home/sanjith/Ai_Engineering/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from wikipediaapi import Wikipedia
wiki = Wikipedia('RAGBot/0.0', 'en')
doc = wiki.page('Jeffrey_Epstein').text
paragraphs = doc.split('\n\n')  # chunking

In [7]:
import textwrap

for i, p in enumerate(paragraphs):
    wrapped_text = textwrap.fill(p, width=100)

    print("-----------------------------------------------------------------")
    print(wrapped_text)
    print("-----------------------------------------------------------------")

-----------------------------------------------------------------
Jeffrey Edward Epstein (January 20, 1953 – August 10, 2019) was an American financier, human
trafficker, child sex offender, and serial rapist. He began his professional career as a teacher
(without a degree) at the Dalton School. After his dismissal from the school in 1976, he entered the
banking and finance sector, working at Bear Stearns in various roles, before starting his own firm.
Epstein cultivated an elite social circle and procured underage girls who were subjected to repeated
rape and sexual violence, by him and his associates. In 2005, police in Palm Beach, Florida, began
investigating Epstein after a parent reported that he had sexually abused her 14-year-old daughter.
Federal officials identified 36 girls, some as young as 14 years old, whom Epstein had allegedly
sexually abused. Epstein pleaded guilty and was convicted in 2008 by a Florida state court of
procuring a child for prostitution and of soliciting

In [8]:
docs_embed = model.encode(paragraphs, normalize_embeddings=True)

In [9]:
docs_embed.shape
docs_embed[0]

array([ 1.06009450e-02, -2.29386371e-02, -1.61199365e-02, -6.50231540e-03,
        1.58579126e-02, -3.40938978e-02,  9.87488329e-02,  6.35555312e-02,
        1.49873579e-02,  1.36030419e-03,  7.41395168e-03, -5.22549003e-02,
        3.43314596e-02,  7.50184758e-03, -1.79798827e-02, -4.26088925e-04,
        9.24138129e-02,  2.87963916e-02, -1.56434886e-02,  8.55509471e-03,
        1.03683490e-02, -2.74819378e-02, -2.98493225e-02,  2.84792073e-02,
       -7.77322380e-03, -3.83960754e-02,  9.52684972e-03,  4.53805588e-02,
       -4.73081134e-03, -4.96611036e-02, -4.49117906e-02, -2.17999462e-02,
        5.82650816e-03, -8.75284709e-03, -2.90548219e-03,  1.92770697e-02,
        2.58364454e-02,  4.95615602e-02, -3.47155631e-02, -1.66356973e-02,
        8.20901245e-03,  2.53661107e-02,  2.91168205e-02,  2.30315942e-02,
       -3.42142880e-02,  2.73392107e-02, -3.77372876e-02, -5.51057234e-02,
       -7.81605579e-03, -3.09003014e-02,  2.02941219e-03, -4.08126116e-02,
        1.32669657e-02, -

In [10]:
query = "When was Epstein arrested again?"
query_embed = model.encode(query, normalize_embeddings=True)

In [11]:
query_embed.shape

(768,)

In [12]:
import numpy as np
similarities = np.dot(docs_embed, query_embed.T)
similarities.shape
print(similarities)

[0.6848847  0.4367706  0.4814813  0.43852746 0.50258356 0.5007763
 0.47099015 0.38316178 0.5776812  0.37452945 0.47481948 0.48917264
 0.46908888 0.44831246 0.44821027 0.5350319  0.49339664 0.50076914
 0.5788058  0.5450331  0.5973271  0.53255    0.5381429  0.5487318
 0.6367149  0.5893075  0.769935   0.65056163 0.6790111  0.5666418
 0.5291959  0.40382668 0.5181074  0.45984745 0.5841621  0.41915494
 0.526351   0.53072643 0.5317155  0.5253131  0.4920764  0.52292395
 0.5016341  0.5339421  0.4089651  0.48011795 0.5288725  0.5190563
 0.49134946 0.48314998 0.54583347 0.55800533 0.49820447 0.4941883
 0.51144886 0.5254498  0.45922348 0.42877942 0.4985416  0.6008873
 0.488688   0.55672705 0.51520497 0.54681826 0.5688851  0.5239823
 0.45123377 0.5483735  0.46654636 0.49432975 0.15687048 0.53977275]


In [13]:
top_3_idx = np.argsort(similarities, axis=0)[-3:][::-1].tolist()

In [14]:
most_similar_documents = [paragraphs[idx] for idx in top_3_idx]

In [15]:
CONTEXT = ""
for i, p in enumerate(most_similar_documents):
  wrapped_text = textwrap.fill(p, width=100)

  print("-----------------------------------------------------------------")
  print(wrapped_text)
  print("-----------------------------------------------------------------")
  CONTEXT += wrapped_text + "\n\n"

-----------------------------------------------------------------
Second set of criminal charges (2019) Sex trafficking charges On July 6, 2019, Epstein was arrested
when he returned to the US from France by the FBI-NYPD Crimes Against Children Task Force at
Teterboro Airport in New Jersey on charges of sex trafficking during the years 2002 to 2005. He was
jailed at the Metropolitan Correctional Center in New York City. According to witnesses and sources
on the day of his arrest, about a dozen FBI agents forced open the door to his Manhattan townhouse,
the Herbert N. Straus House, with search warrants. The search of his townhouse turned up evidence of
sex trafficking and also found "hundreds—and perhaps thousands—of sexually suggestive photographs of
fully—or partially—nude females." Some of the photos were confirmed as those of underage females. In
a locked safe, compact discs were found with handwritten labels including the descriptions: "Young
[Name] + [Name]", "Misc nudes 1", and "

In [16]:
query = "When was Epstein arrested the second time?"

In [17]:
prompt = f"""
use the following CONTEXT to answer the QUESTION at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

CONTEXT: {CONTEXT}
QUESTION: {query}

"""

In [25]:
import os
import openai
from dotenv import load_dotenv

# This function looks for the .env file in your folder
load_dotenv() 

# Now os.environ can find the key
client = openai.OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

In [26]:
response = client.chat.completions.create(
  model="llama-3.1-8b-instant",
  messages=[
    {"role": "user", "content": prompt},
  ]
)

In [27]:
print(response.choices[0].message.content)

Epstein was arrested on July 6, 2019, on sex trafficking charges.
